# 长期记忆
短期记忆记录的是 `会话级别（线程，Thread）` 的数据， 会话间不共享 。
而长期记忆记录的是`用户特定或应用级别`的数据，任何会话都可以随时访问。

长期记忆的分类：
- 语义记忆 Semantic Memory - 存事实
- 情景记忆 Episodic Memory - 存经验
- 程序性记忆 Procedural Memory - 存规则/做事方法

## 存储架构
**长期记忆的存储是 store -> namespace -> key -> value 的四层架构。**
### 第一层Store(记忆仓库)
- Store是 langgraph.store.base.BaseStore 的子类实例，由全类名可知，store是由LangGraph提供的。
    常用实现类：
      - InMemoryStore ：将长期记忆存储在内存，适合测试
      - PostgresStore ：将长期记忆存储在外部的PostgreSQL数据库，适合生产环境

- 开发期可用 InMemoryStore；生产建议数据库后端，如 PostgresStore

### 第二层Namespace(记忆空间)
- 数据类型是由任意长度的 tuple[str, ...] 表示的 `层级路径` 。作用上很像“文件路径 / 文件夹层级”，用于给长期记忆分组和隔离。数据类型为 `字符串元组` 。
  
### 第3层：Key（键）
是该 namespace 下的唯一标识，单条记忆的唯一键，数据类型为 `字符串(str)`

### 第4层：Value（值）
是存储的值，数据类型为 `字典(dict[str, Any])`

In [ ]:
namespace = ("users", "user_123", "preferences") # 元组类型
key = "profile" # 字符串类型
value = { # 字典类型
    "language": "zh-CN",
    "style": "short_direct",
    "likes": ["python", "rag"]
}
store.put(namespace, key, value)

同一个 AI 应用通常会为每个独立会话维护各自的短期状态 State；

而长期记忆通常可以共享同一个 Store 实例，再通过 namespace 区分不同用户、组织、业务域或会话相关数据。例如：

```
AI 应用
├─ thread_id = t1 -> state_1
│  ├─ messages = [
│  │    {"role": "user", "content": "我想去北京旅游"},
│  │    {"role": "assistant", "content": "你想玩几天？"}
│  │  ]
│  ├─ current_intent = "travel_planning"
│  └─ collected_slots = {"destination": "北京"}
│
├─ thread_id = t2 -> state_2
│  ├─ messages = [
│  │    {"role": "user", "content": "帮我写周报"}
│  │  ]
│  ├─ current_intent = "write_report"
│  └─ collected_slots = {}
│
├─ thread_id = t3 -> state_3
│  ├─ messages = [
│  │    {"role": "user", "content": "我喜欢简洁风格的 UI"}
│  │  ]
│  ├─ current_intent = "ui_design"
│  └─ collected_slots = {"style": "minimal"}
│
└─ shared store
   ├─ namespace = (user_1, "memories")
   │  ├─ key = "profile"
   │  │  value = {
   │  │    "name": "张三",
   │  │    "city": "上海",
   │  │    "preferences": ["简洁风格", "中文回复"]
   │  │  }
   │  ├─ key = "travel_preference"
   │  │  value = {
   │  │    "favorite_cities": ["北京", "杭州"],
   │  │    "budget_level": "medium"
   │  │  }
   │  └─ key = "writing_style"
   │     value = {
   │       "tone": "professional",
   │       "language": "zh-CN"
   │     }
   │
   ├─ namespace = (user_2, "memories")
   │  ├─ key = "profile"
   │  │  value = {
   │  │    "name": "李四",
   │  │    "city": "深圳"
   │  │  }
   │  └─ key = "product_interest"
   │     value = {
   │       "topics": ["AI Agent", "RAG", "Workflow"]
   │     }
   │
   └─ namespace = (user_1, thread_3, "artifacts")
      ├─ key = "draft_v1"
      │  value = {
      │    "type": "html",
      │    "content": "<html>...</html>"
      │  }
      ├─ key = "ui_notes"
      │  value = {
      │    "summary": "用户偏好留白多、低饱和配色"
      │  }
      └─ key = "final_scheme"
         value = {
           "palette": ["#F5F1E8", "#1F3A5F", "#D97B2D"],
           "font_style": "clean"
         }
```

In [1]:
# 基于装饰器实现
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

## 基础API的使用
LangChain 1.2.x 的长期记忆基于 store 持久化数据，相关的API有：
- put() ：负责写入
- get() ：负责读取
- search() ：负责检索

我们可以在Agent执行流程之外直接访问长期记忆。